# 🤖 Pre-Workshop Notebook 7: A Primer on AI, Agentic AI and Physical AI
## Scientific Workflows for Brain and Behavioral Research
### National Science Foundation Supported Learning Initiative (Award No. OAC-2417875)

You have used an AI assistant to write code in every notebook so far. This notebook steps back and asks what that assistant is, what makes some AI systems **agentic** (able to act on their own) or **physical** (able to move things in the real world), and how to tell when to trust them. Everything here comes up again in the onsite labs: Lab 2 trains a model, Lab 3 builds a device that takes commands, and the optional Cutebot case puts a decision loop on a small robot car.

No programming experience is needed. Most cells give you a prompt to copy into your AI assistant; a few are ready to run as they are.

***

## 📋 Set Up Your Profile
*Run the cell below to record your progress.*

In [ ]:
# PROFILE SYNC
student_name = "Your Name"
institution = "Your University/Organization"
active_notebook = "Notebook 07 — AI, Agentic AI and Physical AI"

print(f"✅ Profile synced for {student_name} ({institution})")
print(f"Current notebook: {active_notebook}")

***

## 🤖 Using AI Tools to Build Your Workflow ("Ask → Check → Build → Document")

1. **Ask:** copy the suggested prompt into your preferred AI tool (Claude, ChatGPT, Copilot, Gemini…).
2. **Check:** before running anything, ask three questions. *Does it do what I asked? Does it do anything I didn't ask for (downloads, deleting files, sending data anywhere)? Can I explain each step in plain words?* This matters: a 2026 study of six leading AI models found that 25.1% of the code samples they generated contained a confirmed, exploitable security flaw.
3. **Build:** paste the code into the empty cell, run it, and look at the result.
4. **Document:** write in the reflection box what the code did and what it taught you.

> 🔒 **Never paste into a public AI tool:** identifiable participant data, anything covered by your IRB protocol, passwords or access keys. Describe the *shape* of your data ("a table of 14 EEG channels by 38,000 time points") instead of pasting the data itself.

***
## 🧠 Part A: What an AI System Is, and How to Check It
### Topic: Learned vs. Hand-Written Rules, and Testing AI-Written Code

**Artificial intelligence (AI)** is a broad term for computer systems that do tasks we'd normally say need human judgment, such as recognizing a pattern or making a decision. **Machine learning** is the part of AI where the system learns the pattern from example data instead of following rules a person wrote. Your AI coding assistant is a **generative AI** system: a large language model trained on huge amounts of text and code, which writes new text by predicting what should come next. It is often right, but it can also be confidently wrong.

### 🧭 Activity A.1 — A Trained Model vs. a Hand-Written Rule

When people imagine moving their right hand, the "mu" rhythm over the *left* motor cortex (electrode C3) tends to weaken, and vice versa. A brain-computer interface can use this to tell "imagined left" from "imagined right".

> *Copy the prompt below into your preferred AI tool, then paste the generated code into the cell beneath it.*
>
> "Act as an AI instructor for a neuroscience researcher with no programming background. Write Python code using NumPy and scikit-learn that:
> 1. Simulates 200 trials of imagined hand movement (half 'left', half 'right'), each described by two features: mu-rhythm power at electrode C3 and at electrode C4. For 'right' trials C3 power is lower than C4; for 'left' trials the reverse. Add trial-to-trial noise, and make the C3 electrode read about 30% higher than C4 overall because of a different contact quality. Use a random seed of 0.
> 2. Splits the trials into 70% for training and 30% for testing, keeping them separate.
> 3. Applies a hand-written rule to the test trials: 'if C3 power < C4 power, predict right; otherwise predict left'.
> 4. Trains a logistic-regression model on the training trials only, then predicts the test trials.
> 5. Prints the test accuracy of the rule, the trained model, and random guessing (50%).
> Add plain-language comments explaining what 'learned from data' means compared with 'following a rule'."

In [ ]:
# 💻 Paste your AI-generated code here, then run this cell.

### 🧭 Activity A.2 — Checking AI-Written Code with a Known Answer

The most useful single habit for checking AI-written code: **run it on an input where you already know the answer.** If you build a signal yourself, you know exactly what the code should find.

> *Copy the prompt below into your preferred AI tool, then paste the generated code into the cell beneath it.*
>
> "Write a Python function `band_power(signal, fs, low, high)` that returns the average power of `signal` (sampled at `fs` samples per second) between `low` and `high` Hz, using `scipy.signal.periodogram`. Then test it: build a 10-second signal at 250 samples per second that is a 10 Hz sine wave with amplitude 2 plus a 20 Hz sine wave with amplitude 1. Print the alpha-band power (8–13 Hz), the beta-band power (13–30 Hz), and their ratio. Explain in a comment what ratio we should expect and why."

In [ ]:
# 💻 Paste your AI-generated code here, then run this cell.

**Now find a bug.** The cell below is the kind of code an AI assistant can produce: it runs without any error and looks reasonable. Run it on the same test signal. Does it find the 10 Hz rhythm?

In [ ]:
import numpy as np

def band_power_buggy(signal, fs, low, high):
    spectrum = np.abs(np.fft.rfft(signal)) ** 2
    freqs = np.fft.rfftfreq(len(signal))          # <- looks fine, but is it?
    in_band = (freqs >= low) & (freqs <= high)
    return spectrum[in_band].mean() if in_band.any() else 0.0

# The same test signal as above: 10 Hz (amplitude 2) + 20 Hz (amplitude 1), 250 samples per second
fs_test = 250
t_test = np.arange(0, 10, 1 / fs_test)
known_signal = 2 * np.sin(2 * np.pi * 10 * t_test) + np.sin(2 * np.pi * 20 * t_test)

print("Alpha power (buggy version):", band_power_buggy(known_signal, fs_test, 8, 13))
print("Beta power  (buggy version):", band_power_buggy(known_signal, fs_test, 13, 30))

> ✍️ **Your turn:** the buggy version finds no power at all. Paste it into your AI assistant and ask: *"This function returns 0 for a signal I know contains a 10 Hz sine wave at 250 samples per second. What is wrong?"* Write the AI's explanation, and whether you agree, in the reflection below.

### ✍️ Part A Reflection
*Double-click this cell to write your response.*

* **What I observed:** In A.1, which did better on the test trials, the rule or the trained model? Where might the rule still be the safer choice?
* **Checking AI code:** In A.2, what was wrong with the buggy function, and would you have noticed without the known-answer test?
* **Connecting to key concepts:** Activities A.1 and A.2 use **Concepts A1–A7** from the Reference Guide below.

***
## 🦾 Part B: Agentic and Physical AI
### Topic: The Perception–Decision–Action Loop, Human Oversight, and How Agents Fail

An **agentic** system doesn't just answer one question: it senses its situation, decides what to do, acts, then senses the result and repeats. **Physical AI** is AI whose actions move something in the real world, such as a robot, a wheelchair or a stimulator. The lab you'll do in Lab 3, where the cloud sends a device a new setting while it runs, is a small step in this direction.

### 🧭 Activity B.1 — A Perception–Decision–Action Loop

> *Copy the prompt below into your preferred AI tool, then paste the generated code into the cell beneath it.*
>
> "Write a short Python simulation of a small robot car that tries to stay 20 cm from a wall. Each step, it should (1) **sense**: read a distance sensor that returns the true distance plus a little random noise; (2) **decide**: move forward if it is more than 22 cm away, back if it is closer than 18 cm, otherwise stop; (3) **act**: move 2 cm in the chosen direction. Name the three functions `sense`, `decide` and `act`. Start 40 cm from the wall, run 15 steps with a random seed of 1, and print each step's reading, decision and new true distance. Label each part of the loop with a comment, and add a comment marking where a human could review a decision before it is carried out."

In [ ]:
# 💻 Paste your AI-generated code here, then run this cell.

### 🧭 Activity B.2 — When an Agent Acts on Bad Data

Real sensors fail: a loose wire, a reflection, a dropped packet. Run the cell below as given. At step 5 the sensor glitches and reports 2 cm. Watch what the car does.

In [ ]:
import numpy as np

def run(decide_fn, glitch_step=5, steps=10, seed=1):
    # Simulate the robot car with a sensor glitch at `glitch_step`, using your decide function
    STEP = 2.0
    rng_local = np.random.default_rng(seed)
    distance = 20.5
    for step in range(1, steps + 1):
        reading = 2.0 if step == glitch_step else distance + rng_local.normal(0, 0.5)
        decision = decide_fn(reading)
        if decision == "forward":
            distance -= STEP
        elif decision == "back":
            distance += STEP
        flag = "   <-- sensor glitch" if step == glitch_step else ""
        print(f"step {step:2d}: reading {reading:5.1f} cm -> {decision:15s} -> now {distance:4.1f} cm{flag}")

run(decide)

The car believed a broken reading and backed away for no reason. Acting confidently on bad data is one of the most common ways agentic and physical AI systems fail.

> *Copy the prompt below into your preferred AI tool, then paste the generated code into the cell beneath it.*
>
> "Here is my robot's `decide(reading)` function: [paste it]. Write a safer version `decide_safe(reading)` that keeps track of the previous reading and, if a new reading jumps by more than 5 cm from the previous one in a single step, does not move and returns 'ask human' instead, so a person can check the sensor. Then call `run(decide_safe)`."

In [ ]:
# 💻 Paste your AI-generated code here, then run this cell.

### 🧭 Activity B.3 — The Autonomy Spectrum, and Reading the Hype

Systems range from **suggest only** (a person decides everything) through **act with approval** (the system proposes, a person confirms) to **act on its own**. Most real systems sit in between. Pick where each system below sits, then run the cell to check yourself.

In [ ]:
# 🧠 Quick Check — where does each system sit on the autonomy spectrum?
q1_ai_coding_assistant_you_paste_from = "— choose —"  #@param ["— choose —", "Suggest only", "Act with approval", "Act on its own"]
q2_coding_agent_that_runs_code_after_you_click_allow = "— choose —"  #@param ["— choose —", "Suggest only", "Act with approval", "Act on its own"]
q3_lab3_device_applying_a_cloud_threshold = "— choose —"  #@param ["— choose —", "Suggest only", "Act with approval", "Act on its own"]
q4_robot_car_with_ask_human_safeguard = "— choose —"  #@param ["— choose —", "Suggest only", "Act with approval", "Act on its own"]

key = {
    "q1": ("Suggest only", "The assistant writes code, but nothing happens until you choose to paste and run it."),
    "q2": ("Act with approval", "The agent runs code itself, but only after a person approves each action."),
    "q3": ("Act on its own", "In Lab 3 the device changes its own threshold as soon as a valid command arrives; no person confirms it."),
    "q4": ("Act with approval", "It acts on its own most of the time, but hands unusual cases to a person: a mix, closest to 'act with approval' for those cases."),
}
given = {"q1": q1_ai_coding_assistant_you_paste_from, "q2": q2_coding_agent_that_runs_code_after_you_click_allow,
         "q3": q3_lab3_device_applying_a_cloud_threshold, "q4": q4_robot_car_with_ask_human_safeguard}
score = 0
for q, (answer, why) in key.items():
    ok = given[q] == answer
    score += ok
    print(f"{'✅' if ok else '❌'} {q}: {answer}. {why}")
print(f"\nScore: {score}/4 (a self-check, not graded)")

**Reading the numbers together.** In June 2025 the analyst firm Gartner predicted that 33% of enterprise software applications will include agentic AI by 2028, up from less than 1% in 2024, and *also* that over 40% of agentic AI projects will be cancelled by the end of 2027 because of rising costs, unclear value or weak risk controls. Both can be true at once.

> *Ask your AI tool:* "Agentic AI is spreading fast, yet Gartner expects over 40% of agentic AI projects to be cancelled by the end of 2027. For a research lab thinking of using an AI agent to run part of its data analysis, what does that combination suggest about which uses are likely to work and which to avoid?"

> ✍️ Paste the most useful part of the answer, and your own one-sentence view, in the reflection below.

### ✍️ Part B Reflection
*Double-click this cell to write your response.*

* **What I observed:** In B.2, what did the car do with the broken reading before and after the safeguard? What could go wrong with the safeguard itself?
* **Human in the loop:** For a device used with patients or research participants, which decisions would you insist a person approves?
* **Connecting to key concepts:** Part B uses **Concepts A8–A15** from the Reference Guide below.

***

## 📑 What Comes Next

You've finished the pre-workshop preparation. At the onsite workshop, Lab 1 builds a single-participant workflow, Lab 2 trains a model on a whole group (and asks honestly how good it is), and Lab 3 builds a small device that takes commands from the cloud. The optional **Cutebot case** later puts this notebook's perception–decision–action loop on a real or simulated robot car.

***
## 📖 Reference Guide: Key Concepts for AI, Agentic AI and Physical AI

*These are numbered A1–A15 so they don't clash with the 50 concepts in Notebooks 01–06.*

**What AI is**
- **A1. Artificial Intelligence (AI):** computer systems that do tasks we'd normally say need human judgment, such as recognizing a pattern or making a decision.
- **A2. Machine Learning:** the part of AI where a system learns a pattern from example data instead of following rules a person wrote.
- **A3. Training vs. Inference:** training adjusts a model using example data; inference uses the trained model on new data it hasn't seen.
- **A4. Rule-Based vs. Learned Control:** a hand-written "if–then" rule is predictable and needs no data; a learned model can capture patterns no one wrote down but only works for data like its training data.
- **A5. Generative AI and Large Language Models:** systems trained on huge amounts of text and code that produce new text by predicting what comes next. AI coding assistants are built on them.
- **A6. Confidently Wrong Output ("Hallucination"):** AI output that looks plausible but is wrong, such as code that runs without error yet computes the wrong thing, or a function that doesn't exist.
- **A7. Known-Answer Test:** checking code on an input where you already know the right result, such as a signal you built yourself.

**Agentic and physical AI**
- **A8. Agent / Agentic AI:** a system that senses, decides and acts, often repeating that loop on its own. Agency is a matter of degree.
- **A9. Perception–Decision–Action Loop:** the cycle of sensing the environment, choosing an action, acting, and sensing the result.
- **A10. Physical AI (Embodied AI):** AI whose decisions move something in the real world, such as a robot, prosthesis or stimulator.
- **A11. Autonomy Spectrum:** from "suggest only", through "act with approval", to "act on its own". Most real systems sit in between.
- **A12. Human-in-the-Loop:** a person reviews or approves an action before it happens; it trades some speed for safety and accountability.
- **A13. Failure Modes:** common ways agents go wrong, such as acting confidently on bad sensor data or pursuing a goal in a way the designer didn't intend.
- **A14. Growth and Risk Together:** agentic AI is spreading fast *and* many projects fail; both facts matter when deciding whether to use it.
- **A15. Data Privacy with AI Tools:** never paste identifiable participant data, IRB-protected information, passwords or access keys into a public AI tool; describe the data's shape instead.

***
## ✅ Self-Check: What Did You Learn?

| Concept Area | Concept | Where You Demonstrated It |
|---|---|---|
| What AI is | A4 — Rule-based vs. learned control | [ Enter Cell ID ] |
| What AI is | A6 — Confidently wrong output | [ Enter Cell ID ] |
| What AI is | A7 — Known-answer test | [ Enter Cell ID ] |
| Agentic AI | A9 — Perception–decision–action loop | [ Enter Cell ID ] |
| Agentic AI | A12 — Human-in-the-loop | [ Enter Cell ID ] |
| Agentic AI | A13 — Failure modes | [ Enter Cell ID ] |

***
*End of Pre-Workshop Notebook 7 — A Primer on AI, Agentic AI and Physical AI*